# Phase 7 — Chẩn đoán validation trước vòng cải thiện 2

**Notebook mới, chưa chạy GPU.** Chạy ba code cell lần lượt từ trên xuống trên runtime Colab mới.

Mục tiêu: đo P0 và model cũ trên cùng **1.080 ảnh `val_select` (590 real, 490 fake)**, ở hai điều kiện clean và JPEG Q30 sau resize/crop. Kết quả gồm logits từng ảnh, recall/AUC theo nhóm và danh sách lỗi để xác định hướng sửa tiếp.

- Code lấy từ Git nhánh **`hoang`**, khóa commit **`0dabd725e5b7a351514a9b7ee34f4bd6014b584f`** đã kiểm thử.
- Chỉ forward để chẩn đoán; không train, không fit calibration, không chấm CFE/test benchmark.
- Model hiện hành vẫn là gói `24869052…`; P0 chưa được chấp nhận thay thế. G7.2 hiện hành FAIL 6/8, G7.3 pending tham chiếu tốt nhất theo điều kiện.
- Chọn bất kỳ GPU CUDA tương thích và đủ bộ nhớ. Runner probe cả hai model rồi chốt một batch chung; không bắt buộc T4.

**Input Drive cần có:**

```text
MyDrive/Fatformer/
  phase7/train_crossdomain_v2.tar
  phase7/improvement_round1/pilot_20261109_182359/p0_legacy_full/best.pth
  checkpoint/phase7/anchored_v2_run1/fatformer_anchored_v2_best.pth
```

Không sửa notebook đã chạy của các task trước.

## Cell 1 — Kiểm tra GPU

Chọn **Runtime → Change runtime type → GPU** trước khi chạy. Cell này chưa đo chất lượng model.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU trước khi chạy'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f'VRAM trống/tổng: {free_bytes / 1024**3:.2f}/{total_bytes / 1024**3:.2f} GiB')
print('Dung lượng phù hợp sẽ được xác nhận bằng probe cả hai checkpoint trước khi đo.')

## Cell 2 — Clone nhánh `hoang` và khóa code đã kiểm thử

Dùng runtime mới, chưa có `/content/fatformer-xla`. Cell không ghi đè repo đang làm việc. Không `pull` hoặc đổi commit giữa phiên chạy.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/fatformer-xla')
PINNED_COMMIT = '0dabd725e5b7a351514a9b7ee34f4bd6014b584f'

assert not REPO.exists(), 'Repo đã tồn tại. Giữ log cũ và dùng runtime mới để tránh ghi đè.'
subprocess.run([
    'git', 'clone', '--branch', 'hoang', '--single-branch',
    'https://github.com/LeeVHoangtk3/fatformer-xla.git', str(REPO)
], check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO, check=True)
subprocess.run([
    'git', 'merge-base', '--is-ancestor', PINNED_COMMIT, 'origin/hoang'
], cwd=REPO, check=True)
DIAG_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True
).strip()
assert DIAG_COMMIT == PINNED_COMMIT, 'Commit thực tế không khớp bản đã khóa'
assert (REPO / 'notebooks/task/phase7_validation_diagnostics/start_colab.py').is_file()
print('Commit đã khóa:', DIAG_COMMIT)

## Cell 3 — Mount Drive và đo validation

Chấp nhận yêu cầu kết nối Drive khi Colab hiển thị. Bước đọc/hash archive và copy weights lên SSD có thể mất vài phút.

Runner sẽ kiểm hash ảnh/checkpoint, probe bộ nhớ, đo hai model trên clean và Q30, rồi xuất kết quả. Q30 chạy đủ 1.080 ảnh và giữ riêng 300 ảnh lịch sử, seed 42. Dùng chung operating point chọn model lịch sử **T=1,6402, threshold=0,9127**; đây không phải calibration deployment riêng của mỗi checkpoint.

Nếu lỗi/OOM: giữ notebook và log, dùng runtime mới; không đổi batch/commit trong cùng run. Không chạy lại Cell 2 để ghi đè repo.

In [ ]:
DIAG_REPO = str(REPO)
DIAG_MAX_BATCH = 16

exec(compile(
    (REPO / 'notebooks/task/phase7_validation_diagnostics/start_colab.py').read_text(encoding='utf-8'),
    'start_colab.py', 'exec'
))

## Sau khi hoàn tất

Kết quả được lưu tại:

```text
MyDrive/Fatformer/phase7/improvement_round2/diagnostics/<DIAG_RUN_ID>/
  identity.json
  batch_contract.json
  status.json
  candidate_p0/predictions_clean.npz
  candidate_p0/predictions_q30_post_crop.npz
  incumbent/predictions_clean.npz
  incumbent/predictions_q30_post_crop.npz
  analysis/diagnosis.json
  analysis/group_metrics.csv
  analysis/errors.csv
  validation_diagnostics_without_weights.zip
```

ZIP tải về là **ZIP kết quả**, không phải ZIP code; không chứa weights. Có thể chép trực tiếp thư mục kết quả Drive thay ZIP.

1. Lưu notebook đã chạy thành `notebooks/task/phase7_validation_diagnostics/phase7_validation_diagnostics_executed.ipynb`.
2. Đưa kết quả giải nén vào `wait/phase7_validation_diagnostics/<DIAG_RUN_ID>/`.
3. Hậu kiểm dự đoán từng ảnh trước khi chọn thí nghiệm train tiếp. Chưa chạy lại 7.14/7.15 hoặc thay model hiện hành ở bước này.

**Giới hạn diễn giải:** `val_select` đã được dùng chọn P0, nên đây là chẩn đoán, không phải blind test mới. Real90 tính trên validation không tạo ngưỡng deployment. AUC của một nhóm fake dùng chung 590 real reference; nhóm ít ảnh có độ bất định lớn. Notebook hoàn tất không có nghĩa G7.2/G7.3 PASS.